# Práctica 2 · Actividad 2: Data Profiling de Telco Customer Churn

En este notebook hacemos el *perfilado de datos* (data profiling) del dataset **Telco Customer Churn**: 7,043 clientes de una empresa de telecomunicaciones, con datos de sus servicios, su contrato y lo que pagan, y una última columna, `Churn`, que indica si el cliente terminó cancelando el servicio. La idea es conocer bien los datos antes de pensar en modelos: qué tipo de variables hay, si falta información, si hay valores raros y si algo podría confundir a un modelo.

Seguimos la tabla de la Actividad 2. Para las variables **numéricas** y **categóricas** revisamos el tipo de dato, los ausentes, los valores únicos, los estadísticos, la moda y el dominio. Además buscamos filas duplicadas, columnas constantes, categorías inconsistentes, identificadores, correlaciones altas y posibles fugas de información. Debajo de cada tabla explicamos con palabras sencillas qué significan los números, y al final juntamos los hallazgos más importantes para el producto parcial.

**Cómo usarlo:** ejecuta las celdas de arriba hacia abajo (Shift + Enter).

> Declaración de uso de IA generativa: este código se generó con asistencia de Claude (Anthropic). Ayudó a mejorar el estilo visual de las tablas y, en lugar de entregar solo un script, sugirió usar un notebook para mostrar mejor la información solicitada.

## 0. Instalación (solo la primera vez)
Si ya tienes `pandas`, `numpy` y `jinja2` instalados, puedes saltarte esta celda. `jinja2` es lo que permite darles formato a las tablas.

In [ ]:
%pip install pandas numpy jinja2

## 1. Librerías, carga de datos y configuración
Aquí importamos las librerías y cargamos el CSV. Hay un detalle con `TotalCharges`: pandas la lee como texto porque 11 filas traen un espacio en blanco en lugar de un número. Es un ausente "disfrazado" que pandas no reconoce como `NaN`, así que primero lo contamos y después convertimos la columna a número (lo que no se pueda convertir queda como ausente).

También definimos qué columnas son numéricas, cuáles categóricas, cuál es el identificador y cuál es la variable objetivo (`Churn`).

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# --- Carga de datos (carpeta datasets/ junto a este notebook) ---
df = pd.read_csv("datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# TotalCharges se lee como texto porque 11 filas traen un espacio en blanco (ausente oculto).
# Primero los contamos y luego convertimos: errors="coerce" vuelve ausente (NaN) lo que no es número.
print("Tipo de TotalCharges antes de convertir:", df["TotalCharges"].dtype)
blancos_total = int((df["TotalCharges"].astype(str).str.strip() == "").sum())
print("Valores en blanco en TotalCharges:", blancos_total)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# --- Configuración del dataset ---
OBJETIVO = "Churn"
POSITIVA = "Yes"
IDENTIFICADORES = ["customerID"]
NUMERICAS = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAS = [c for c in df.columns if c not in NUMERICAS + IDENTIFICADORES + [OBJETIVO]]
FRECUENCIAS = ["SeniorCitizen", "Contract", "InternetService", "MultipleLines", "PaymentMethod"]
DISCRETAS = []                              # no hay numéricas discretas relevantes
COLS_CORR = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
UMBRAL_CORR = 0.7

df.head()

Antes de convertir, `TotalCharges` venía como texto y con 11 valores en blanco. Eso explica por qué pandas no la reconoció como numérica: bastó un espacio vacío en unas pocas filas para que toda la columna se leyera como texto. Con `errors="coerce"` esos 11 espacios se vuelven ausentes de verdad (`NaN`), y ya podemos tratarlos como tales en el resto del análisis.

### Estilo de las tablas
Esta celda solo define el formato de las tablas (líneas finas y un par de colores suaves para resaltar lo que conviene revisar). No hace ningún análisis, así que no hay nada que interpretar.

In [ ]:
# --- Estilo común de las tablas (simple y limpio) ---
GRIS = "rgba(128,128,128,0.45)"      # líneas del encabezado
SUAVE = "rgba(128,128,128,0.18)"     # líneas entre filas
AMARILLO = "background-color:#FFF3CD;color:#000"   # para revisar
ROJO = "background-color:#F8D7DA;color:#000"       # ausencia alta

ESTILO = [
    {"selector": "table", "props": [("border-collapse", "collapse"), ("font-size", "14px")]},
    {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"),
                                       ("font-size", "15px"), ("font-weight", "bold"), ("padding", "6px 0")]},
    {"selector": "th", "props": [("padding", "6px 12px"), ("text-align", "left")]},
    {"selector": "td", "props": [("padding", "6px 12px"), ("border-bottom", f"1px solid {SUAVE}"),
                                 ("max-width", "560px"), ("white-space", "normal")]},
    {"selector": "th.col_heading", "props": [("border-bottom", f"2px solid {GRIS}")]},
    {"selector": "th.blank", "props": [("border-bottom", f"2px solid {GRIS}")]},
    {"selector": "th.row_heading", "props": [("border-bottom", f"1px solid {SUAVE}"), ("font-weight", "600")]},
]

def bonito(df, titulo):
    """Aplica el estilo común y un título a una tabla."""
    d = df.copy()
    d.index.name = None          # evita una fila extra con el nombre del índice
    return d.style.set_caption(titulo).set_table_styles(ESTILO)

def aplicar(styler, funcion, subset):
    """Aplica una función de color celda por celda (compatible con pandas viejos y nuevos)."""
    if hasattr(styler, "map"):
        return styler.map(funcion, subset=subset)
    return styler.applymap(funcion, subset=subset)

def color_ausencia(v):
    """Sin color: sin ausentes. Amarillo: menos de 30%. Rojo: 30% o más."""
    if v == 0:
        return ""
    return AMARILLO if v < 30 else ROJO

def color_alerta(v):
    """Amarillo para 'Sí' o para cualquier número mayor que cero."""
    if v == "Sí" or (isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) and v > 0):
        return AMARILLO
    return ""

def titulo(texto):
    display(Markdown(texto))


## 2. Análisis solicitado en la práctica

### 2.1 Tipo de dato, valores válidos y ausentes
*(numéricas: sí · categóricas: sí)*

Empezamos por ver cómo reconoció pandas cada columna: qué tipo de dato le asignó, cuántos valores válidos (no nulos) tiene y cuántos le faltan.

In [ ]:
def tipo_valores_ausentes(df):
    """Tipo de dato, clase de variable, valores no nulos (válidos) y ausentes por columna."""
    def clase(c):
        if c == OBJETIVO: return "Objetivo"
        if c in IDENTIFICADORES: return "Identificador"
        if c in NUMERICAS: return "Numérica"
        return "Categórica"
    t = pd.DataFrame({
        "Tipo de dato": df.dtypes.astype(str),
        "Clase": [clase(c) for c in df.columns],
        "Valores no nulos": df.notna().sum(),
        "Ausentes": df.isna().sum(),
    })
    return bonito(t, "Tipo de dato y valores ausentes").format(
        {"Valores no nulos": "{:,}", "Ausentes": "{:,}"})

tipo_valores_ausentes(df)

Casi todo viene completo: 20 de las 21 columnas no tienen ningún ausente. La excepción es `TotalCharges`, con 11 ausentes (7,032 valores válidos de 7,043).

Hay dos columnas cuyo tipo de dato puede engañar. `SeniorCitizen` aparece como entero, pero en realidad es una categoría (0 = no, 1 = sí), por eso la tratamos como categórica. Y `customerID` es texto, pero la marcamos como identificador. Con esto queda claro que el tipo de dato solo dice cómo guardó pandas la columna, no necesariamente lo que significa la variable.

### 2.2 Porcentaje de ausencia y valores únicos
*(numéricas: sí · categóricas: sí)*

Aquí medimos qué porcentaje de cada columna está vacío. Las celdas con ausentes se resaltan: amarillo si falta menos del 30% y rojo si falta el 30% o más. También contamos los valores únicos, que sirven para distinguir categóricas de continuas y para detectar identificadores (una columna con casi todos los valores distintos suele ser uno).

In [ ]:
def ausencia_y_unicos(df):
    t = pd.DataFrame({
        "% de ausencia": (df.isna().mean() * 100).round(2),
        "Valores únicos": df.nunique(),
        "% de únicos": (df.nunique() / len(df) * 100).round(1),
    })
    est = bonito(t, "Ausencia y valores únicos")
    est = est.format({"% de ausencia": "{:.2f}%", "Valores únicos": "{:,}", "% de únicos": "{:.1f}%"})
    return aplicar(est, color_ausencia, ["% de ausencia"])

ausencia_y_unicos(df)

Solo `TotalCharges` tiene ausentes, y es apenas el 0.16%, así que el dataset está prácticamente completo.

En valores únicos hay dos casos que destacan. `customerID` tiene 100% de únicos: cada cliente tiene el suyo, o sea que es un identificador. `TotalCharges` tiene 92.7%, lo cual es normal en un monto acumulado de dinero, donde casi nadie coincide con otro al centavo. Por su parte, `tenure` solo tiene 73 valores distintos porque se mide en meses (de 0 a 72), y las columnas de Sí/No tienen apenas 2 o 3.

### 2.3 Mínimo, máximo, media, mediana y desviación estándar
*(solo numéricas)*

Un resumen rápido de la escala y el centro de cada variable numérica. Comparar la media con la mediana da una primera pista de cómo se reparten los datos.

In [ ]:
def resumen_numericas(df, cols):
    t = pd.DataFrame({
        "Mínimo": df[cols].min(),
        "Máximo": df[cols].max(),
        "Media": df[cols].mean(),
        "Mediana": df[cols].median(),
        "Desviación estándar": df[cols].std(),
    })
    return bonito(t, "Resumen de variables numéricas").format("{:,.2f}")

resumen_numericas(df, NUMERICAS)

`tenure` va de 0 a 72 meses (seis años), con media de 32.4 y mediana de 29. Como están cerca, los clientes se reparten de forma bastante pareja. En `MonthlyCharges` pasa lo contrario: la media (64.76) queda por debajo de la mediana (70.35), lo que sugiere un grupo de clientes con cargos muy bajos que jala el promedio hacia abajo. Lo más probable es que sean los clientes que solo contratan teléfono, sin internet.

`TotalCharges` es la que más se despega: su media (2,283) es bastante mayor que su mediana (1,397), y su desviación estándar (2,267) es casi igual a la media. Esto apunta a una cola larga hacia valores altos: pocos clientes, antiguos y con planes caros, acumulan montos muy grandes.

### 2.4 Percentiles y asimetría
*(solo numéricas)*

Los percentiles muestran cómo se reparten los datos (por ejemplo, el P25 es el valor por debajo del cual está el 25% de los clientes). La asimetría resume la forma de la distribución: cerca de 0 es simétrica, y mayor a 1 en valor absoluto indica una cola larga con probables valores extremos. Si la media queda muy por encima de la mediana, la asimetría es positiva (cola hacia la derecha).

In [ ]:
def percentiles_asimetria(df, cols):
    t = df[cols].quantile([.01, .05, .25, .5, .75, .95, .99]).T
    t.columns = ["P1", "P5", "P25", "P50", "P75", "P95", "P99"]
    t["Asimetría"] = df[cols].skew()
    def interpretar(a):
        fuerza = "simétrica" if abs(a) < 0.5 else ("asimetría moderada" if abs(a) < 1 else "asimetría fuerte")
        if fuerza == "simétrica":
            return fuerza
        return fuerza + (" a la derecha" if a > 0 else " a la izquierda")
    t["Interpretación"] = t["Asimetría"].apply(interpretar)
    est = bonito(t, "Percentiles y asimetría").format("{:,.2f}", subset=t.columns[:-1])
    def color_asim(v):
        return AMARILLO if abs(v) > 1 else ""
    return aplicar(est, color_asim, ["Asimetría"])

percentiles_asimetria(df, NUMERICAS)

`tenure` (0.24) y `MonthlyCharges` (−0.22) son prácticamente simétricas. `TotalCharges` tiene una asimetría de 0.96, moderada y hacia la derecha, y muy cerca del 1 que usamos como límite para hablar de asimetría fuerte. Los percentiles lo confirman: el P50 es 1,397, pero el P95 llega a 6,924. Si el modelo es sensible a la escala, conviene tenerlo en cuenta.

Otro detalle: en `tenure`, tanto el P95 como el P99 valen 72, es decir que más del 5% de los clientes está justo en el máximo. Entonces hay un tope en la escala, no una cola de valores que siga creciendo.

### 2.5 Moda y frecuencias por categoría
*(categóricas: sí · numéricas: opcional)*

Para las categóricas primero identificamos la categoría más común (la moda) y qué tan dominante es. Después mostramos las frecuencias completas de las columnas que nos parecieron más importantes.

In [ ]:
def resumen_moda(df, cols):
    filas = []
    for c in cols:
        vc = df[c].value_counts()
        filas.append({"Columna": c, "Categorías": df[c].nunique(), "Moda": vc.index[0],
                      "Frecuencia de la moda": int(vc.iloc[0]),
                      "% de la moda": round(vc.iloc[0] / len(df) * 100, 2)})
    t = pd.DataFrame(filas).set_index("Columna")
    est = bonito(t, "Moda de las variables categóricas")
    est = est.format({"Categorías": "{:,}", "Frecuencia de la moda": "{:,}", "% de la moda": "{:.2f}%"})
    return est

def frecuencias(df, col, top=10):
    """Frecuencia y porcentaje por categoría (incluye ausentes). Muestra las 'top' más frecuentes."""
    vc = df[col].value_counts(dropna=False)
    t = pd.DataFrame({"Categoría": vc.index.astype(str), "Frecuencia": vc.values,
                      "%": (vc.values / len(df) * 100).round(2)}).head(top).set_index("Categoría")
    sufijo = f" (top {top} de {len(vc)})" if len(vc) > top else ""
    est = bonito(t, f"Frecuencias de {col}{sufijo}")
    est = est.format({"Frecuencia": "{:,}", "%": "{:.2f}%"})
    return est

display(resumen_moda(df, CATEGORICAS))
for c in FRECUENCIAS:
    display(frecuencias(df, c))

La mayoría de las variables está bastante repartida, pero algunas están muy cargadas hacia un lado: el 90.3% de los clientes tiene servicio telefónico (`PhoneService`), el 83.8% no es adulto mayor (`SeniorCitizen`) y el 70.0% no tiene dependientes (`Dependents`).

En las frecuencias llama la atención el tipo de contrato: más de la mitad de los clientes (55.0%) está en contrato **mes a mes**, y solo el 20.9% tiene uno de un año y el 24.1% uno de dos. La fibra óptica es el tipo de internet más común (44.0%) y el cheque electrónico el método de pago más usado (33.6%). Más adelante se confirma que `Contract` es justamente la variable más ligada a `Churn`.

### 2.6 Dominio y valores potencialmente inválidos
*(numéricas: sí · categóricas: sí)*

Para cada columna definimos qué valores tienen sentido (su **dominio**) y contamos cuántos quedan fuera. Después listamos otros valores que no son inválidos, pero que conviene revisar.

In [ ]:
SI_NO = ["Yes", "No"]
SERVICIO = ["Yes", "No", "No internet service"]
DOMINIO = [
    ("gender", "Male o Female", lambda s: s.notna() & ~s.isin(["Male", "Female"])),
    ("SeniorCitizen", "0 o 1", lambda s: s.notna() & ~s.isin([0, 1])),
    ("Partner", "Yes o No", lambda s: s.notna() & ~s.isin(SI_NO)),
    ("Dependents", "Yes o No", lambda s: s.notna() & ~s.isin(SI_NO)),
    ("tenure", "entero entre 0 y 72 meses", lambda s: s.notna() & ((s < 0) | (s > 72))),
    ("PhoneService", "Yes o No", lambda s: s.notna() & ~s.isin(SI_NO)),
    ("MultipleLines", "Yes, No o No phone service", lambda s: s.notna() & ~s.isin(["Yes", "No", "No phone service"])),
    ("InternetService", "DSL, Fiber optic o No", lambda s: s.notna() & ~s.isin(["DSL", "Fiber optic", "No"])),
    ("OnlineSecurity", "Yes, No o No internet service", lambda s: s.notna() & ~s.isin(SERVICIO)),
    ("OnlineBackup", "Yes, No o No internet service", lambda s: s.notna() & ~s.isin(SERVICIO)),
    ("DeviceProtection", "Yes, No o No internet service", lambda s: s.notna() & ~s.isin(SERVICIO)),
    ("TechSupport", "Yes, No o No internet service", lambda s: s.notna() & ~s.isin(SERVICIO)),
    ("StreamingTV", "Yes, No o No internet service", lambda s: s.notna() & ~s.isin(SERVICIO)),
    ("StreamingMovies", "Yes, No o No internet service", lambda s: s.notna() & ~s.isin(SERVICIO)),
    ("Contract", "Month-to-month, One year o Two year", lambda s: s.notna() & ~s.isin(["Month-to-month", "One year", "Two year"])),
    ("PaperlessBilling", "Yes o No", lambda s: s.notna() & ~s.isin(SI_NO)),
    ("PaymentMethod", "4 métodos de pago", lambda s: s.notna() & ~s.isin(["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"])),
    ("MonthlyCharges", "mayor que 0", lambda s: s.notna() & (s <= 0)),
    ("TotalCharges", ">= 0", lambda s: s.notna() & (s < 0)),
]

SOSPECHOSOS = [
    ("TotalCharges", "Ausente (venía como espacio en blanco)", df["TotalCharges"].isna(), "Ausentes ocultos: Pandas los leyó como texto"),
    ("tenure", "tenure igual a 0", df["tenure"] == 0, "Clientes nuevos, sin ningún mes facturado todavía"),
    ("TotalCharges", "TotalCharges ausente con tenure = 0", df["TotalCharges"].isna() & (df["tenure"] == 0), "Los 11 ausentes coinciden con clientes nuevos (no son errores de captura)"),
]

In [ ]:
def revisar_dominio(df, reglas):
    """reglas: lista de (columna, dominio esperado en texto, función que devuelve la máscara de valores inválidos)."""
    filas = []
    for col, dominio, invalido in reglas:
        s = df[col]
        es_num = pd.api.types.is_numeric_dtype(s) and col in NUMERICAS
        if es_num:
            observado = f"{s.min():,.2f} a {s.max():,.2f}"
        else:
            vals = sorted(s.dropna().astype(str).unique())
            observado = ", ".join(vals[:6]) + (f" … ({len(vals)} en total)" if len(vals) > 6 else "")
        filas.append({"Columna": col, "Dominio esperado": dominio, "Valores observados": observado,
                      "Valores fuera del dominio": int(invalido(s).sum())})
    t = pd.DataFrame(filas).set_index("Columna")
    est = bonito(t, "Dominio de cada columna")
    return aplicar(est, color_alerta, ["Valores fuera del dominio"])

revisar_dominio(df, DOMINIO)

In [ ]:
def valores_sospechosos(df, criterios):
    """Valores que no son inválidos pero conviene revisar. criterios: (columna, criterio, máscara, comentario)."""
    filas = [{"Columna": c, "Criterio": crit, "Filas": int(m.sum()), "Comentario": com}
             for c, crit, m, com in criterios]
    t = pd.DataFrame(filas)   # una misma columna puede aparecer en varias filas
    est = bonito(t, "Valores a revisar").hide(axis="index")
    return aplicar(est, color_alerta, ["Filas"])

valores_sospechosos(df, SOSPECHOSOS)

Ninguna columna tiene valores fuera de su dominio: todas las categorías son las esperadas y los números están en rangos razonables. Por eso la columna de valores fuera del dominio sale en ceros.

Lo interesante está en la tabla de valores a revisar: las 11 filas con `TotalCharges` ausente son exactamente las 11 filas con `tenure = 0`. Eso cambia la lectura, porque ya no parecen errores de captura. Son clientes recién dados de alta que todavía no reciben su primera factura, así que su total acumulado aún no existe. Lo más natural es rellenarlos con 0 en lugar de eliminar las filas.

## 2.7 Revisiones adicionales solicitadas

### Filas duplicadas
Las contamos de dos maneras: con todas las columnas y sin los identificadores. Lo hacemos así porque un identificador único hace que ninguna fila parezca repetida, aunque el resto de los datos sea idéntico.

**Ojo:** al quitar los identificadores pueden aparecer coincidencias que no son registros repetidos, sino clientes distintos con las mismas características. Por eso conviene revisarlas antes de eliminarlas.

In [ ]:
def filas_duplicadas(df):
    sin_id = df.drop(columns=IDENTIFICADORES)
    t = pd.DataFrame({"Filas": {
        "Duplicadas (todas las columnas)": int(df.duplicated().sum()),
        "Duplicadas (sin considerar identificadores)": int(sin_id.duplicated().sum()),
    }})
    t["% del total"] = (t["Filas"] / len(df) * 100).round(2)
    est = bonito(t, "Filas duplicadas").format({"Filas": "{:,}", "% del total": "{:.2f}%"})
    return aplicar(est, color_alerta, ["Filas"])

filas_duplicadas(df)

Para entender mejor esas coincidencias, miramos de cerca las filas repetidas (sin el identificador):

In [ ]:
repetidas = df[df.drop(columns=IDENTIFICADORES).duplicated(keep=False)]
vistazo = pd.DataFrame({
    "Filas involucradas": [len(repetidas)],
    "tenure mínimo": [int(repetidas["tenure"].min())],
    "tenure máximo": [int(repetidas["tenure"].max())],
    "% con contrato mes a mes": [(repetidas["Contract"] == "Month-to-month").mean() * 100],
}, index=["Filas repetidas"])
bonito(vistazo, "Un vistazo a las filas repetidas (sin identificador)").format({"% con contrato mes a mes": "{:.0f}%"})

Con todas las columnas no hay ninguna fila duplicada (0), algo esperable porque `customerID` es distinto en cada fila. Al quitar el identificador aparecen 22 filas repetidas (0.31%), que involucran 42 registros si contamos cada fila junto con su copia.

Al revisarlas de cerca, todas son clientes con `tenure = 1` y contrato mes a mes. Con tantas columnas categóricas es normal que clientes nuevos que contrataron el mismo plan básico coincidan en todo lo demás. Por eso **no las eliminamos**: no hay evidencia de que sean registros capturados dos veces.

### Columnas constantes o casi constantes
Una columna donde casi todas las filas tienen el mismo valor casi no ayuda a distinguir a unos clientes de otros. Como criterio, la consideramos casi constante si su valor más frecuente concentra más del 99% de las filas.

In [ ]:
def columnas_constantes(df, umbral=0.99):
    """Una columna es casi constante si su valor más frecuente concentra más del 99% de las filas."""
    t = pd.DataFrame({
        "Valor más frecuente": [str(df[c].value_counts().index[0]) for c in df.columns],
        "% de filas con ese valor": [round(df[c].value_counts(normalize=True).iloc[0] * 100, 2) for c in df.columns],
    }, index=df.columns)
    t["Constante o casi constante"] = np.where(t["% de filas con ese valor"] / 100 > umbral, "Sí", "No")
    t = t.sort_values("% de filas con ese valor", ascending=False)
    est = bonito(t, f"Columnas constantes o casi constantes (umbral: {umbral:.0%})")
    est = est.format({"% de filas con ese valor": "{:.2f}%"})
    return aplicar(est, color_alerta, ["Constante o casi constante"])

columnas_constantes(df)

Ninguna columna es constante ni casi constante. La más concentrada es `PhoneService`, con 90.3% de "Yes", todavía lejos del 99% del umbral. Las columnas que aparecen al final (`tenure`, `MonthlyCharges`, `TotalCharges`, `customerID`) tienen porcentajes cercanos a cero porque casi cada valor es distinto, no porque sean constantes.

Un dato importante sobre la variable objetivo: `Churn` está desbalanceada, con 73.46% de "No" y 26.54% de "Sí". No es un problema de calidad de los datos, pero hay que tenerlo presente al entrenar y evaluar un modelo.

### Categorías inconsistentes
Primero revisamos el formato: espacios sobrantes, mayúsculas o guiones que harían distintas a categorías que en realidad son la misma. Después buscamos niveles que, aunque estén bien escritos, repiten información que ya tenemos en otra columna.

In [ ]:
def categorias_inconsistentes(df, cols):
    """Busca categorías que serían la misma tras quitar espacios, mayúsculas o guiones."""
    filas = []
    for c in cols:
        s = df[c].dropna().astype(str)
        normal = s.str.strip().str.lower().str.replace(r"[\s_\-]+", " ", regex=True)
        filas.append({"Columna": c, "Categorías distintas": s.nunique(),
                      "Distintas tras normalizar": normal.nunique(),
                      "Valores con espacios sobrantes": int((s != s.str.strip()).sum())})
    t = pd.DataFrame(filas).set_index("Columna")
    t["Inconsistencias"] = np.where((t["Categorías distintas"] != t["Distintas tras normalizar"]) |
                                    (t["Valores con espacios sobrantes"] > 0), "Sí", "No")
    est = bonito(t, "Revisión de formato en las categorías")
    return aplicar(est, color_alerta, ["Inconsistencias"])

categorias_inconsistentes(df, CATEGORICAS)

In [ ]:
# Niveles redundantes: estas categorías repiten información que ya dan PhoneService o InternetService
redundantes = []
for c in CATEGORICAS:
    for nivel, columna_origen, valor_origen in [("No internet service", "InternetService", "No"),
                                                 ("No phone service", "PhoneService", "No")]:
        n = int((df[c] == nivel).sum())
        if n > 0 and c != columna_origen:
            redundantes.append({"Columna": c, "Nivel redundante": nivel, "Filas": n,
                                "Equivale a": f"{columna_origen} == {valor_origen}"})
t = pd.DataFrame(redundantes).set_index("Columna")
est = bonito(t, "Niveles redundantes: se pueden tratar como \"No\"").format({"Filas": "{:,}"})
aplicar(est, color_alerta, ["Filas"])

El formato está limpio: no hay espacios sobrantes ni variantes de mayúsculas, por eso la primera tabla marca "No" en todas las columnas.

El hallazgo está en la segunda tabla. La categoría "No internet service" aparece 1,526 veces en seis columnas (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV` y `StreamingMovies`), y es justo el número de clientes con `InternetService = No`. Lo mismo pasa con "No phone service" en `MultipleLines` (682 filas), que equivale a `PhoneService = No`. No es un error, pero esos niveles repiten algo que ya sabemos por otra columna. Para modelar se pueden tratar como "No".

### Identificadores únicos
Buscamos columnas con (casi) todos los valores distintos. Un identificador sirve para nombrar a un cliente, no para describirlo, así que no aporta patrones generales y se deja fuera del modelo.

In [ ]:
def identificadores(df, umbral=95):
    t = pd.DataFrame({
        "Valores únicos": df.nunique(),
        "% de únicos": (df.nunique() / len(df) * 100).round(2),
    })
    t["Único por fila"] = np.where(t["Valores únicos"] == len(df), "Sí", "No")
    t = t[t["% de únicos"] > umbral].sort_values("% de únicos", ascending=False)
    if t.empty:
        print("No hay columnas con más de", umbral, "% de valores únicos.")
        return None
    est = bonito(t, "Posibles identificadores")
    est = est.format({"Valores únicos": "{:,}", "% de únicos": "{:.2f}%"})
    return aplicar(est, color_alerta, ["Único por fila"])

identificadores(df)

Solo `customerID` supera el 95% de valores únicos, y es único en las 7,043 filas. Sirve para nombrar al cliente pero no lo describe, así que un modelo no puede aprender nada general de él (incluso podría terminar memorizando clientes individuales). Lo excluimos del análisis y del modelo.

### Correlaciones elevadas
Calculamos la correlación de Pearson entre las variables numéricas. El umbral `UMBRAL_CORR` define qué consideramos alto. Cuando dos variables están muy correlacionadas, aportan información parecida, es decir, redundante.

In [ ]:
def color_correlacion(v):
    if pd.isna(v):
        return ""
    intensidad = int(abs(v) * 150)
    if v >= 0:
        return f"background-color:rgb(255,{255 - intensidad},{255 - intensidad});color:#000"
    return f"background-color:rgb({255 - intensidad},{255 - intensidad},255);color:#000"

def matriz_correlacion(df, cols):
    corr = df[cols].corr()
    est = bonito(corr, "Correlación de Pearson (rojo: positiva, azul: negativa)").format("{:.2f}")
    return aplicar(est, color_correlacion, None)

def pares_correlacion_alta(df, cols, umbral=UMBRAL_CORR):
    corr = df[cols].corr()
    pares = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(corr.columns)
             for b in corr.columns[i + 1:] if abs(corr.loc[a, b]) >= umbral]
    if not pares:
        print(f"No hay pares de variables con |r| >= {umbral}.")
        return None
    t = pd.DataFrame(pares, columns=["Variable A", "Variable B", "r"]).set_index("Variable A")
    return bonito(t, f"Correlaciones elevadas (|r| >= {umbral})").format({"r": "{:.3f}"})

display(matriz_correlacion(df, COLS_CORR))
altas = pares_correlacion_alta(df, COLS_CORR)
if altas is not None:
    display(altas)

El único par por encima del umbral de 0.7 es `tenure` con `TotalCharges` (r = 0.826), y tiene todo el sentido: mientras más meses lleva un cliente, más dinero acumula. `MonthlyCharges` y `TotalCharges` también se relacionan (0.65), aunque sin llegar al umbral, y `SeniorCitizen` casi no se correlaciona con nada (valores entre 0.02 y 0.22).

Cuando dos variables dicen casi lo mismo, a algunos modelos les cuesta separar el efecto de cada una (multicolinealidad). Más adelante habrá que decidir si se conservan las dos o solo una.

### 2.8 Posibles fugas de información
Hay **fuga** cuando una variable le da al modelo la respuesta, o una pista casi directa, de una forma que no tendríamos al predecir en la vida real. Puede pasar porque la variable se genera después del resultado, porque se deriva del propio objetivo o porque identifica de forma única a cada registro.

La tabla de abajo calcula tres señales automáticas:
- **Identificador:** la columna es única por fila y no sirve para generalizar.
- **Asociación alta (≥ 0.5):** una variable muy ligada al objetivo merece revisión, pero **no es fuga por sí sola**: puede ser un buen predictor que sí conocemos antes del resultado.
- **Ausencia informativa:** que falte el dato se relaciona con el objetivo (una diferencia de 10 puntos porcentuales o más).

Las señales solo avisan. El criterio final lo ponemos nosotros con una pregunta sencilla: ¿tendríamos esa variable disponible en el momento real de decidir?

In [ ]:
def cramers_v(x, y):
    """Asociación entre dos variables categóricas (0 = ninguna, 1 = total)."""
    t = pd.crosstab(x, y)
    n = t.values.sum()
    esperado = np.outer(t.sum(axis=1), t.sum(axis=0)) / n
    chi2 = ((t.values - esperado) ** 2 / esperado).sum()
    k = min(t.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else np.nan

def senales_fuga(df):
    """Para cada variable calcula: asociación con el objetivo y si la ausencia del dato se relaciona con el objetivo."""
    y = (df[OBJETIVO] == POSITIVA).astype(int)
    filas = []
    for c in df.columns:
        if c == OBJETIVO:
            continue
        pct_aus = df[c].isna().mean() * 100
        metrica, asoc = "—", np.nan
        if c in IDENTIFICADORES:
            pass
        elif pd.api.types.is_numeric_dtype(df[c]) and (c in NUMERICAS or df[c].nunique() <= 10):
            asoc, metrica = abs(df[c].corr(y)), "|r|"
        elif df[c].nunique() <= 50:
            asoc, metrica = cramers_v(df[c].fillna("(ausente)"), y), "V de Cramér"
        # Diferencia en la tasa del objetivo entre filas con dato ausente y con dato presente
        dif = np.nan
        if pct_aus >= 5:
            dif = (y[df[c].isna()].mean() - y[df[c].notna()].mean()) * 100
        señales = []
        if c in IDENTIFICADORES:
            señales.append("Identificador: excluir del modelo")
        if not np.isnan(asoc) and asoc >= 0.5:
            señales.append("Asociación alta con el objetivo: revisar")
        if not np.isnan(dif) and abs(dif) >= 10:
            señales.append("La ausencia del dato se relaciona con el objetivo")
        if c not in IDENTIFICADORES and metrica == "—":
            señales.append("Muchas categorías: asociación no calculada")
        filas.append({"Variable": c, "% de ausencia": round(pct_aus, 2), "Métrica": metrica,
                      "Asociación con el objetivo": asoc,
                      "Dif. si falta el dato (pp)": dif,
                      "Señal": "; ".join(señales) if señales else "—"})
    return pd.DataFrame(filas).set_index("Variable")

senales = senales_fuga(df)
est = bonito(senales, "Señales de posible fuga de información")
est = est.format({"% de ausencia": "{:.2f}%", "Asociación con el objetivo": "{:.3f}",
                  "Dif. si falta el dato (pp)": "{:+.1f}"}, na_rep="—")
est = aplicar(est, lambda v: AMARILLO if any(k in v for k in ("Identificador", "Asociación alta", "ausencia")) else "", ["Señal"])
est

Ninguna variable llega a 0.5 de asociación con `Churn`, y la única columna con ausentes, `TotalCharges`, no alcanza el 5% que pide la regla para evaluar la ausencia informativa. La única bandera de la tabla es `customerID`, por ser un identificador.

Las asociaciones más fuertes son las de `Contract` (V de Cramér de 0.41), `tenure` (|r| de 0.35), `OnlineSecurity` (0.35) y `TechSupport` (0.34). Son variables razonables, que se conocen antes de que el cliente decida irse y que no delatan la respuesta, así que las tomamos como predictores legítimos y no como fuga. En el otro extremo, `gender` (0.009) y `PhoneService` (0.012) casi no tienen relación con la cancelación.

**Evidencia para el criterio.** Dos variables merecen una revisión más de cerca: `tenure` (la antigüedad del cliente en meses) y `TotalCharges` (lo que ha pagado en total).

In [ ]:
# Evidencia 1: tenure queda fijado en la fecha de baja para quienes se fueron
t1 = df.groupby("Churn")["tenure"].agg(["count", "mean", "median"])
t1.columns = ["Clientes", "tenure promedio (meses)", "tenure mediana (meses)"]
display(bonito(t1, "tenure según Churn: los clientes que se fueron tienen menos meses acumulados")
        .format({"Clientes": "{:,}", "tenure promedio (meses)": "{:.1f}", "tenure mediana (meses)": "{:.0f}"}))

# Evidencia 2: TotalCharges es casi tenure x MonthlyCharges (variable derivada / redundante)
aprox = df["tenure"] * df["MonthlyCharges"]
dif = (df["TotalCharges"] - aprox).abs()
t2 = pd.DataFrame({"Valor": {
    "Correlación tenure vs TotalCharges": df["tenure"].corr(df["TotalCharges"]),
    "Correlación MonthlyCharges vs TotalCharges": df["MonthlyCharges"].corr(df["TotalCharges"]),
    "Diferencia absoluta mediana |TotalCharges − tenure × MonthlyCharges|": dif.median(),
}})
bonito(t2, "TotalCharges es casi una función de tenure y MonthlyCharges").format("{:.3f}")

Los clientes que se fueron (`Churn = Yes`) tienen un `tenure` promedio de 18.0 meses y una mediana de 10, frente a 37.6 y 38 de los que se quedaron. O sea, quienes se van suelen hacerlo pronto. Aquí hay que tener cuidado: en quien ya se fue, el `tenure` queda congelado en su mes de baja, así que en parte refleja el resultado. En cambio, para un cliente activo sí es un dato disponible (su antigüedad hasta hoy). Por eso la mantenemos, pero con cautela.

Sobre `TotalCharges`: su correlación es 0.826 con `tenure` y 0.651 con `MonthlyCharges`, y la diferencia mediana respecto a `tenure × MonthlyCharges` es de unos 28.65. Es casi, aunque no exactamente, el producto de las otras dos (el precio mensual pudo haber cambiado con el tiempo). En la práctica aporta poca información nueva, así que conviene valorar si vale la pena conservarla.

## 2.9 Producto parcial: hallazgos más importantes
Esta tabla se arma sola a partir de todo lo que calculamos arriba y junta los hallazgos principales en un solo lugar. Sirve como punto de partida para el producto parcial.

In [ ]:
HALLAZGOS_EXTRA = [
    ("TotalCharges", f"{blancos_total} valores venían como espacio en blanco (texto) y todos corresponden a clientes con tenure = 0."),
    ("Niveles redundantes", "\"No internet service\" (6 columnas) y \"No phone service\" repiten InternetService = No y PhoneService = No."),
    ("TotalCharges y tenure", f"Correlación de {df['tenure'].corr(df['TotalCharges']):.3f}: TotalCharges es casi tenure × MonthlyCharges (variable derivada)."),
]

def hallazgos(df):
    h = []
    aus = (df.isna().mean() * 100).round(2)
    aus = aus[aus > 0].sort_values(ascending=False)
    if len(aus):
        h.append(("Valores ausentes", "; ".join(f"{c}: {v:.2f}%" for c, v in aus.items())))
    h.append(("Filas duplicadas", f"{int(df.duplicated().sum())} (todas las columnas); "
                                  f"{int(df.drop(columns=IDENTIFICADORES).duplicated().sum())} sin considerar identificadores"))
    casi = [c for c in df.columns if df[c].value_counts(normalize=True).iloc[0] > 0.99]
    h.append(("Columnas constantes o casi constantes", ", ".join(casi) if casi else "Ninguna"))
    ids = [c for c in df.columns if df[c].nunique() / len(df) > 0.95]
    h.append(("Identificadores únicos", ", ".join(ids) if ids else "Ninguno"))
    asim = df[NUMERICAS].skew()
    fuertes = asim[asim.abs() > 1].sort_values(key=abs, ascending=False)
    h.append(("Asimetría fuerte (|asimetría| > 1)",
              "; ".join(f"{c}: {v:.2f}" for c, v in fuertes.items()) if len(fuertes) else "Ninguna"))
    corr = df[COLS_CORR].corr()
    pares = [f"{a}–{b}: {corr.loc[a, b]:.3f}" for i, a in enumerate(corr.columns)
             for b in corr.columns[i + 1:] if abs(corr.loc[a, b]) >= UMBRAL_CORR]
    h.append((f"Correlaciones elevadas (|r| >= {UMBRAL_CORR})", "; ".join(pares) if pares else "Ninguna"))
    marcadas = senales[senales["Señal"].str.contains("Identificador|Asociación alta|ausencia", regex=True)]
    if len(marcadas):
        h.append(("Señales de posible fuga", "; ".join(f"{c} ({s})" for c, s in marcadas["Señal"].items())))
    h.extend(HALLAZGOS_EXTRA)
    t = pd.DataFrame(h, columns=["Tema", "Detalle"]).set_index("Tema")
    return bonito(t, "Producto parcial: hallazgos más importantes")

hallazgos(df)

Si hubiera que quedarse con lo más importante para el producto parcial, serían tres cosas: los 11 ausentes ocultos de `TotalCharges` (que en realidad son clientes nuevos), la relación casi directa entre `TotalCharges` y `tenure`, y la presencia de `customerID` como identificador junto con los niveles redundantes de "No internet service" y "No phone service". Fuera de eso, el dataset está limpio y sin problemas graves.